# Solutions · Chapter 03-06 · Lines, slopes, and logarithms

E9 and E16 are the two worth attempting first - E9 shows a log-log fit failing in a way that is easy
to miss, and E16 settles how to compare models that live on different scales.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

temperature = np.array([12, 15, 18, 21, 24, 27])
rentals = np.array([180, 225, 270, 315, 360, 405])
print("temperature", temperature)
print("rentals    ", rentals)

## E1 · The slope in a sentence, and what the intercept would need

**The slope:** *"Each additional degree Celsius is associated with 15 more rentals per day."* Units:
rentals per degree.

**For the intercept to be a real claim**, 0 degrees would have to be a temperature the data actually
contains, and the relationship would have to be the same there as it is between 12 and 27. Neither
holds: the coldest observed day is 12 degrees, and freezing weather affects cycling through a
mechanism - ice, darkness, clothing - that has nothing to do with the gentle warm-weather trend the
six days describe.

Note also the weaker word in that sentence. **"Associated with", not "causes"** - 02-07's rule. Six
warm days do not establish that warming a day would produce fifteen more rentals; hot days differ
from cool ones in more ways than temperature.

## E2 · Why centring moves the intercept and not the slope

The slope is a **ratio of changes** - rise over run. Sliding every x value left by the same amount
changes neither the rises nor the runs, so the ratio is untouched.

The intercept is the value **at a particular place**, namely x = 0. Centring moves where that place
is: it used to be 0 degrees, and after subtracting the mean it is the mean temperature. The line has
not moved; the label on the origin has.

## E3 · Two coefficient readings

- **`log(y) = a + bx`**: one more unit of x multiplies y by `exp(b)` - a constant *percentage*
  change per unit of x.
- **`y = a + b log(x)`**: each doubling of x adds `b x ln(2)` to y - a constant *absolute* change
  per doubling, so the effect of x has diminishing returns.

## E4 · `revenue = 200 + 35 x staff`

In [ ]:
print("slope     : 35 euros of daily revenue per additional member of staff")
print("intercept : 200 euros of daily revenue with zero staff")
print("prediction at 0 staff: %d euros" % (200 + 35 * 0))

**Is 200 euros at zero staff credible? Occasionally yes, and that is what makes it worth checking
rather than dismissing.**

- If the business has online orders, vending machines or standing subscriptions, then "revenue with
  nobody working" is a real quantity and 200 euros may be a genuine estimate.
- If it is a shop that cannot open without staff, the intercept is bookkeeping. And the model is
  probably being fitted over a range like 3 to 12 staff, so zero is an extrapolation of the same kind
  as 0 degrees.

**The test is the same one every time: is x = 0 inside the data, and is the mechanism the same
there?** For staff counts the first is nearly always no.

## E5 · A coefficient of 0.35 on logged sales

In [ ]:
b = 0.35
print("naive reading : %.0f%%" % (100 * b))
print("correct       : exp(%.2f) - 1 = %.2f%%" % (b, 100 * (np.exp(b) - 1)))
print("understated by: %.2f percentage points" % (100 * (np.exp(b) - 1) - 100 * b))

**41.91%, not 35%** - the naive reading understates the promotion's effect by 6.91 percentage points,
about a fifth of the reported figure.

## E6 · Doubling every seven years

In [ ]:
years_to_double = 7
rate = np.log(2) / years_to_double
print("the coefficient on a natural-log fit : %.5f per year" % rate)
print("the annual growth factor             : %.5f" % np.exp(rate))
print("as a percentage                      : %.2f%% per year" % (100 * (np.exp(rate) - 1)))

**10.41% a year**, and the fitted coefficient would be **0.09902**.

Worth noticing how close 0.09902 and 0.1041 are - at growth rates this small the naive reading is off
by four hundredths of a percentage point, which is why the shortcut survives. There is also a
well-known mental version of this relationship: **70 divided by the percentage growth rate gives the
doubling time**, because `ln(2) = 0.693`. At 10% a year, seven years.

## E7 · `describe_line`

In [ ]:
def describe_line(x, y, x_units, y_units):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    slope, intercept = np.polyfit(x, y, 1)
    slope, intercept = round(float(slope), 9) + 0.0, round(float(intercept), 9) + 0.0

    print("Each additional %s is associated with a change of %.4f %s." % (x_units, slope, y_units))
    print("At %s = 0 the line predicts %.4f %s." % (x_units, intercept, y_units))
    if not (x.min() <= 0 <= x.max()):
        print("  WARNING: 0 %s is outside the observed range (%.2f to %.2f)."
              % (x_units, x.min(), x.max()))
        print("           The intercept positions the line; it is not a finding.")
        print("           Centred, the intercept would be %.4f %s at the average %s of %.2f."
              % (np.polyfit(x - x.mean(), y, 1)[1], y_units, x_units, x.mean()))


describe_line(temperature, rentals, "degree Celsius", "rentals")
print()
describe_line(np.array([-3, -1, 0, 2, 4]), np.array([10, 14, 16, 20, 24]),
              "degree of temperature change", "index points")

The second case has 0 inside its range, so no warning fires and the intercept of 16 is a genuine
prediction about a real, observed situation.

## E8 · Fitting a line to something exponential

In [ ]:
noise_rng = np.random.default_rng(2)
month = np.arange(13)
observed = 40 * 1.25 ** month * noise_rng.lognormal(0, 0.06, 13)

linear = np.polyfit(month, observed, 1)
logged = np.polyfit(month, np.log(observed), 1)

print("straight line on the raw counts : slope %.2f, intercept %.2f" % (linear[0], linear[1]))
print("straight line on log(counts)    : slope %.4f -> growth factor %.4f"
      % (logged[0], np.exp(logged[0])))
print()
print("prediction for month 24, linear fit : %8.1f" % (linear[1] + linear[0] * 24))
print("prediction for month 24, log fit    : %8.1f" % np.exp(logged[1] + logged[0] * 24))
print("the truth                           : %8.1f" % (40 * 1.25 ** 24))
print()
print("residuals of the linear fit:")
print(np.round(observed - (linear[1] + linear[0] * month), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
future = np.arange(0, 25)
ax.plot(month, observed, "o", color="black", label="observed", zorder=3)
ax.plot(future, linear[1] + linear[0] * future, color="#D55E00", label="straight line on counts")
ax.plot(future, np.exp(logged[1] + logged[0] * future), color="#0072B2", label="straight line on log")
ax.plot(future, 40 * 1.25 ** future, "--", color="grey", label="the truth")
ax.set_xlabel("month")
ax.set_ylabel("e-bikes")
ax.legend()
ax.set_title("Both fit the first year. Only one is right about the second")
plt.tight_layout()
plt.show()

**The log fit is far better, and the linear fit's failure is visible without looking at the future.**

Look at the residuals: `+77, +44, +15, -20, -20, -40, -63, -54, -53, -49, +17, +37, +109`. They are
**positive at both ends and negative in the middle** - a clean U shape. That pattern is the signature
of fitting a straight line to a curve, and it is why residual plots are worth drawing: the model is
wrong in a *structured* way, which noise never is.

At month 24 the linear fit says **960** and the log fit says **8,776**, against a truth of **8,470**.
The linear fit is not slightly wrong; it is wrong by a factor of nine, and it was wrong for a reason
that was visible in the residuals a year earlier.

## E9 · A power law, and a power law plus a constant

In [ ]:
power_rng = np.random.default_rng(5)
x = np.linspace(1, 50, 60)

pure = 3 * x ** 1.5 * power_rng.lognormal(0, 0.05, 60)
shifted = 3 * x ** 1.5 * power_rng.lognormal(0, 0.05, 60) + 20

print("y = 3 x^1.5            -> log-log slope %.4f" % np.polyfit(np.log(x), np.log(pure), 1)[0])
print("y = 3 x^1.5 + 20       -> log-log slope %.4f" % np.polyfit(np.log(x), np.log(shifted), 1)[0])
print()
print("at x = 1  : 3 x 1^1.5 = %5.1f   with the offset: %5.1f" % (3, 3 + 20))
print("at x = 50 : 3 x 50^1.5 = %5.0f   with the offset: %5.0f" % (3 * 50 ** 1.5, 3 * 50 ** 1.5 + 20))

The pure power law returns **1.5021** - the exponent, recovered. Adding a constant of 20 drops it to
**1.1683**, which is wrong by 20%.

**Why:** a power law plus a constant is not a power law, and taking logs no longer straightens it. The
offset dominates at small x - 23 against 3, so the value is nearly eight times what the power law
alone would give - and is negligible at large x, where 1,081 against 1,061 is a 2% difference. So the
log-log plot is flat at the left and correctly sloped at the right, and a straight line through it
splits the difference.

**The general point, which applies to every transform in this chapter:** logging linearises a
*specific* functional form. `y = a x^b` becomes a straight line; `y = a x^b + c` does not, and neither
does `y = a e^(bx) + c`. When a log-log fit returns an exponent that drifts as you change the range of
x, an additive constant is the usual culprit - and the fix is to estimate it rather than to log
harder.

## E10 · "A 62% increase"

In [ ]:
print("reported : 62%")
print("correct  : exp(0.62) - 1 = %.1f%%" % (100 * (np.exp(0.62) - 1)))
print("the paper understates its own finding by %.1f percentage points"
      % (100 * (np.exp(0.62) - 1) - 62))

**It should say 85.9%.** The abstract understates the paper's own result by 23.9 percentage points -
a finding roughly 39% larger than the one reported.

This error is common and it is worth being generous about the reason: the shortcut is taught, it is
accurate for the small coefficients most papers report, and nobody rechecks it when a coefficient
comes back large. It remains an error, and it is the one direction of mistake nobody catches in
review, because a smaller claimed effect never triggers scepticism.

## E11 · Rooms at 15,000 and area at 900

**Reason one: the units are different.** 15,000 is euros *per room* and 900 is euros *per square
metre*. Adding one room is not the same intervention as adding one square metre - a room is perhaps
fifteen square metres, so on a like-for-like basis the area coefficient corresponds to about 13,500
euros per room-sized amount of space. The comparison as stated is between two different questions.

**Reason two: the two features are strongly correlated**, and in a model containing both, each
coefficient means *"holding the other constant"*. The rooms coefficient is the value of adding a room
**without adding any floor area** - that is, of subdividing existing space - which is a strange
quantity and not what anyone means by "rooms matter". When features overlap, individual coefficients
become hard to interpret and unstable, which is the subject of multicollinearity in module 05.

A third, if you want it: neither coefficient carries any information about how much each feature
*varies* in the data. A feature with a large coefficient that barely varies explains nothing.

## E12 · `np.log(counts)` producing `-inf`

**The cause: some counts are zero, and `log(0)` is negative infinity.** Not an error, not a warning
by default - just an infinite value that propagates into every downstream mean, fit and metric until
something raises.

**Two fixes, and what each does to the meaning:**

1. **`np.log1p(counts)`, which computes `log(1 + x)`.** Zeros map to 0 and the transform is smooth.
   The cost: the coefficient is no longer a clean elasticity, because the relationship between
   `log1p(x)` and `log(x)` differs most exactly where the counts are small - which is often where the
   interesting rows are. For large counts the two agree closely.
2. **Drop or separate the zeros.** Model "did it happen at all" and "how much, given it happened" as
   two questions. The cost is complexity, and the benefit is that both parts are interpretable and the
   zeros stop being a numerical inconvenience and become a finding.

**And a third response, which is often correct: do not log it.** Zeros in a count usually mean
something - the shop was closed, the sensor was off, nobody came - and 02-04's lesson applies: an
awkward value is more often a message than an obstacle.

## E13 · When to log-transform

> "I log a variable when its effects are naturally multiplicative rather than additive - money,
> populations, counts that grow by percentages - because a model that adds effects will fit such data
> badly at both ends of the range. I also log when a variable spans several orders of magnitude, since
> without it the largest handful of values dominate the fit entirely. I would not log a variable that
> is already on a bounded, meaningful scale - a percentage, a rating out of five, a temperature -
> where the untransformed units are what stakeholders think in and the range is too narrow to matter.
> The cost is that every coefficient changes meaning, so I have to convert back with `exp(b) - 1` to
> report anything, and predictions need care because the mean of the logs is not the log of the mean.
> The decision is really about which scale the *effects* are additive on, not about making a histogram
> look nicer."

The last sentence is what an interviewer is listening for.

## E14 · The server model

In [ ]:
intercept_log, per_user = 2.1, 0.004
for users in [0, 100, 500, 1000]:
    print("%4d concurrent users -> %8.1f ms" % (users, np.exp(intercept_log + per_user * users)))
print()
print("each additional user multiplies response time by exp(%.3f) = %.5f" % (per_user, np.exp(per_user)))
print("each additional 100 users multiplies it by exp(0.4) = %.3f" % np.exp(0.4))
print("one second is crossed at %.0f users" % ((np.log(1000) - intercept_log) / per_user))

**In plain words:** response time grows by a constant *percentage* per user, not a constant number of
milliseconds. Every extra hundred users multiplies the response time by about **1.49** - so the model
says the system degrades geometrically, and the pain arrives suddenly rather than gradually.

**8.2 ms with nobody on it, 12.2 ms at 100 users, 60.3 ms at 500, and one second at 1,202 users.**

The practical reading: the difference between 100 and 500 users is 48 milliseconds and invisible; the
difference between 1,000 and 1,200 is the difference between acceptable and unusable. A capacity plan
based on extrapolating the *linear-looking* early region will be badly wrong, which is E8's failure in
production.

## E15 · For the manager

> "The chart's vertical axis is spaced by multiplication rather than addition - each step up is ten
> times the one below, not ten thousand more. On that kind of axis, steady percentage growth shows up
> as a straight line, and a line that keeps rising at the same angle means we are growing at exactly
> the same rate as before. The reason it looks like it is flattening is that the same *percentage* is
> a bigger and bigger *number* of users, so the curve on a normal chart would be getting steeper. The
> log chart is the one that would bend downwards if we were genuinely slowing."

87 words, and it answers the question underneath the question, which is "are we slowing down".

## E16 · Comparing four models honestly

In [ ]:
compare_rng = np.random.default_rng(11)
X = np.linspace(1, 60, 200)
Y = 5 * X ** 0.7 * compare_rng.lognormal(0, 0.15, 200)

train, test = slice(0, 150), slice(150, 200)


def mae(actual, predicted):
    return float(np.mean(np.abs(actual - predicted)))


results = {}
p = np.polyfit(X[train], Y[train], 1)
results["linear         y = a + bx"] = p[1] + p[0] * X[test]

p = np.polyfit(X[train], np.log(Y[train]), 1)
results["logged outcome log(y) = a + bx"] = np.exp(p[1] + p[0] * X[test])

p = np.polyfit(np.log(X[train]), Y[train], 1)
results["logged input   y = a + b log(x)"] = p[1] + p[0] * np.log(X[test])

power = np.polyfit(np.log(X[train]), np.log(Y[train]), 1)
results["log-log        log(y) = a + b log(x)"] = np.exp(power[1] + power[0] * np.log(X[test]))

for name, predictions in results.items():
    print("%-38s MAE on the original scale of y: %6.3f" % (name, mae(Y[test], predictions)))
print()
print("the log-log fit recovered an exponent of %.4f, against the true 0.7" % power[0])

**The log-log fit wins - MAE 8.699 against 10.135 for linear, 14.755 for a logged input and 50.765
for a logged outcome** - and it recovers the true exponent, 0.6979 against 0.7.

Note that the logged-outcome model is *by far* the worst, despite being the transform people reach for
most often. The data is a power law, not exponential growth, so `log(y) = a + bx` is fitting the wrong
shape and its errors explode at the top of the range.

**Why comparing them in their own transformed spaces would be meaningless.** Two of these models are
fitted to `y` and two to `log(y)`. Their residuals are therefore in different units - bikes against
log-bikes - and an error of 0.1 in log space corresponds to a completely different real error
depending on where you are on the curve. A model fitted on logs will nearly always show a smaller
"error" in its own space, simply because logging compresses the scale, and that number says nothing
about whether its predictions are better.

**The rule this establishes, and it applies throughout the rest of the course:** transform whatever
you like while fitting, but **bring every prediction back to the original units before comparing
anything**. The scale on which you evaluate is the scale the decision is made on, and it is a
different choice from the scale on which you fit.

## Where to go next

**03-07 · Vectors, distance, and matrices.** Everything here used one input. The next chapter is the
notation for many at once - which is what makes distance, similarity and every geometric method in
the course expressible, and what makes the shapes in an error message readable.